### Test WhisperX installation ###

In [1]:
import sys
import os
from pathlib import Path
import logging
import gc
import glob
from multiprocessing import Process

# PyTorch
import torch

# WhisperX
import whisperx
from whisperx.diarize import DiarizationPipeline

logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)

%load_ext autoreload
%autoreload 2
import audiotranscription
print(f'Authors: {audiotranscription.__authors__}')
print(f'Project version: {audiotranscription.__version__}')
print(f'Python version:  {sys.version}')
print(f'PyTorch version: {torch.__version__} with CUDA: {torch.cuda.is_available()}')

Authors: The Core for Computational Biomedicine at Harvard Medical School
https://dbmi.hms.harvard.edu/about-dbmi/core-computational-biomedicine
Project version: v0.0.1
Python version:  3.12.11 (main, Sep 30 2025, 10:25:27) [GCC 14.2.0]
PyTorch version: 2.8.0+cu128 with CUDA: True


In [12]:
# Parameters and files
device = "cuda"
batch_size = 16 # reduce if low on GPU mem
compute_type = "float16" # change to "int8" if low on GPU mem (may reduce accuracy)
whisperx_model = 'small.en'
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

# Directories
project_name = 'proj1'
input_dir = os.path.join(os.environ['DATA_INPUT_DIR'], project_name)
output_dir = os.path.join(os.environ['DATA_OUTPUT_DIR'], f'{project_name}_output')
Path(output_dir).mkdir(parents=True, exist_ok=True)

# Select a file
file_list = glob.glob(os.path.join(input_dir, '*.m4a'))
file = file_list[-1]
print(file)

/app/data/input/proj1/compose.m4a


In [13]:
# Load the model
model = whisperx.load_model(whisperx_model, device, compute_type=compute_type)
# Load the audio file
audio = whisperx.load_audio(file)

2026-03-10 17:25:07 - whisperx.vads.pyannote - INFO - Performing voice activity detection using Pyannote...


Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.1. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint ../../usr/local/lib/python3.12/site-packages/whisperx/assets/pytorch_model.bin`


In [8]:
# Release GPU memory. This does not relase all of the memory.
# We can use the multiprocessing library to accomplish that (see below)
gc.collect()
torch.cuda.empty_cache()

In [14]:
# Transcribe
result_transcription = model.transcribe(audio, batch_size=batch_size)

In [10]:
print(result_transcription.keys())
print(len(result_transcription['segments']))
print(len(result_transcription['language']))
print(result_transcription['language'])

dict_keys(['segments', 'language'])
105
2
en


In [11]:
# Align whisper output, word-level output
model_a, metadata = whisperx.load_align_model(language_code=result_transcription['language'], device=device)
result_words = whisperx.align(result_transcription['segments'], model_a, metadata, audio, device, return_char_alignments=False)
print(result_words.keys())

dict_keys(['segments', 'word_segments'])


In [12]:
# Assign speaker labels
diarize_model = DiarizationPipeline(token=os.environ['HF_TOKEN'], device=device)

2026-03-10 16:40:54 - whisperx.diarize - INFO - Loading diarization model: pyannote/speaker-diarization-community-1


In [13]:
# add min/max number of speakers if known
diarize_segments = diarize_model(audio)
# diarize_model(audio, min_speakers=min_speakers, max_speakers=max_speakers)
result = whisperx.assign_word_speakers(diarize_segments, result_words)

/usr/local/lib/python3.12/site-packages/pyannote/audio/models/blocks/pooling.py:103: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /pytorch/aten/src/ATen/native/ReduceOps.cpp:1839.)
  std = sequences.std(dim=-1, correction=1)


### Use a Python process ###
This completely releases the GPU memory after execution

In [3]:
def run_transcription():
    file = os.path.normpath('/app/data/input/proj1/compose.m4a')
    model = whisperx.load_model(whisperx_model, device, compute_type=compute_type)
    audio = whisperx.load_audio(file)
    result_transcription = model.transcribe(audio, batch_size=batch_size)
    model_a, metadata = whisperx.load_align_model(language_code=result_transcription['language'], device=device)
    result_words = whisperx.align(result_transcription['segments'], model_a, metadata, audio, device, return_char_alignments=False)
    print(result_words['word_segments'][:3])
    # Release GPU memory
    gc.collect()
    torch.cuda.empty_cache()

In [4]:
P = Process(target=run_transcription)

In [ ]:
P.start()